# Run COMPASS survival pipeline locally

This notebook runs the shared COMPASS PROFILE survival pipeline for the
**treatment anchor** (ARPI/androgen-axis + taxanes + radium-223 -- "tx_anchor")
-> `prediction_inputs_tx_anchor/` and `local_runs_tx_anchor/`. The treatment
anchor (first ARPI/taxane/radium-223 exposure) is the sole treatment index for
the COMPASS pipeline and defines time 0: every duration (`t_lab`, `t_platinum`,
...) is measured from it, so the landmark is a pure offset from the anchor
(`--anchor-col none`) with no separate anchor column.

Full processing from the raw OncDRS pull runs in two steps:
1. `compile_COMPASS_cohort_data.py` builds the ICD-C61 cohort source tables
   (text/ICD/health/meds/labs/somatic/PSA/platinum) plus the ARPI/chemo-anchored
   `prostate_arpi_survival_cohort.csv`.
2. `longitudinal_data_processing.py` consumes those tables to build the
   row-level longitudinal prediction data.

Run the shared setup and preprocessing cells first, then the treatment-anchor
section.

## Shared setup

In [ ]:
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path("/data/gusev/USERS/jpconnor/code/CAIA")
SURVIVAL_DIR = PROJECT_ROOT / "COMPASS" / "survival_analysis"
DATA_PREPROCESSING_DIR = PROJECT_ROOT / "COMPASS" / "data_preprocessing"

NEPC_PROJ_PATH = Path("/data/gusev/USERS/jpconnor/data/CAIA/COMPASS/")
CONSOLIDATED_LABS = NEPC_PROJ_PATH / "consolidated_longitudinal_data.csv"
DATA = NEPC_PROJ_PATH / "longitudinal_prediction_data.csv"
SURVIVAL_OUTPUT_ROOT = NEPC_PROJ_PATH / "survival_analysis"

PYTHON = sys.executable
LANDMARKS = [0, 90]
N_FOLDS = 5
FORCE_RERUN = True

# COMPASS durations (t_lab, t_platinum, ...) are measured from the treatment
# anchor (first ARPI/taxane/radium-223 exposure = time 0), so anchor_col is
# "none": the landmark is a pure offset from the anchor with no anchor column.
RUNS = [
    {
        "label": "tx_anchor",
        "title": "Treatment anchor",
        "anchor_col": "none",
        "inputs_dir": SURVIVAL_OUTPUT_ROOT / "prediction_inputs_tx_anchor",
        "output_dir": SURVIVAL_OUTPUT_ROOT / "local_runs_tx_anchor",
    },
]

os.chdir(PROJECT_ROOT)
for run in RUNS:
    run["inputs_dir"].mkdir(parents=True, exist_ok=True)
    run["output_dir"].mkdir(parents=True, exist_ok=True)

for v in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[v] = "1"

print("python:            ", PYTHON)
print("cwd:               ", os.getcwd())
print("survival_dir:      ", SURVIVAL_DIR)
print("data_preprocessing:", DATA_PREPROCESSING_DIR)
print("consolidated_labs: ", CONSOLIDATED_LABS)
print("longitudinal_data: ", DATA)
for run in RUNS:
    print(f"{run['label']:16s}: inputs={run['inputs_dir']} outputs={run['output_dir']}")

## 1. Compile COMPASS cohort data

In [ ]:
!{PYTHON} {DATA_PREPROCESSING_DIR}/compile_COMPASS_cohort_data.py

## 2. Preprocess raw labs once (longitudinal_data_processing.py)

In [ ]:
!{PYTHON} {DATA_PREPROCESSING_DIR}/longitudinal_data_processing.py

## 3. Verify lab filtering

In [ ]:
df = pd.read_csv(CONSOLIDATED_LABS, low_memory=False)
print(f"consolidated rows: {len(df):,}")
print()

oor = df.loc[df["conversion_status"] == "out_of_physiologic_range", "collapsed_measurement"]
print("out_of_physiologic_range rows by collapsed_measurement (top 15):")
print(oor.value_counts().head(15).to_string() if not oor.empty else "  (none -- either no outliers or filter not active)")
print()

for lab, bound, unit in [("Testosterone", 2000, "ng/dL"), ("PSA", 10000, "ng/mL")]:
    sub = df.loc[
        (df["collapsed_measurement"] == lab) & df["numeric_result_standardized"].notna(),
        "numeric_result_standardized",
    ]
    if sub.empty:
        print(f"  {lab}: no standardized values present")
        continue
    over = int((sub > bound).sum())
    print(f"  {lab:13s}: n={len(sub):>6,}  max={sub.max():.2f} {unit}  #>{bound}={over}  (should be 0)")


## Shared run helpers

In [ ]:
TASKS = [
    ("univariate",   0,  "both",      "cox_agg_univariate_nobs_adjusted.csv"),
    ("univariate",   90, "both",      "cox_agg_univariate_nobs_adjusted.csv"),
    ("elastic-net",  0,  "both",      "cox_agg_multivariable_metrics.csv"),
    ("elastic-net",  90, "both",      "cox_agg_multivariable_metrics.csv"),
    ("elastic-net",  0,  "baseline",  "cox_agg_baseline_metrics.csv"),
    ("elastic-net",  90, "baseline",  "cox_agg_baseline_metrics.csv"),
    ("xgboost",      0,  "both",      "landmark_xgboost_metrics.csv"),
    ("xgboost",      90, "both",      "landmark_xgboost_metrics.csv"),
    ("xgboost",      0,  "baseline",  "landmark_xgboost_baseline_metrics.csv"),
    ("xgboost",      90, "baseline",  "landmark_xgboost_baseline_metrics.csv"),
]


def model_output_dir(model):
    return "cox" if model in ("univariate", "elastic-net") else "xgboost"


def clear_prediction_inputs(inputs_dir):
    for pattern in (
        "aggregated_landmark*.csv",
        "pre_treatment_lab_long_landmark*.csv",
    ):
        for p in inputs_dir.glob(pattern):
            p.unlink()
            print(f"  removed {p.name}")
    for fname in (
        "canonical_labs_train_val.csv",
        "build_manifest.json",
        "split_assignments.csv",
        "landmark_mrn_availability.csv",
        "landmark_attrition.json",
    ):
        p = inputs_dir / fname
        if p.exists():
            p.unlink()
            print(f"  removed {p.name}")


def build_prediction_inputs(run):
    print(f"\n========== build inputs: {run['title']} ==========")
    clear_prediction_inputs(run["inputs_dir"])
    cmd = [
        PYTHON, str(DATA_PREPROCESSING_DIR / "build_prediction_inputs.py"),
        "--data", str(DATA),
        "--output-dir", str(run["inputs_dir"]),
        "--anchor-col", run["anchor_col"],
        "--landmark-days", *[str(lm) for lm in LANDMARKS],
        "--time-unit-days", "7",
        "--test-frac", "0.20",
        "--val-frac", "0.20",
        "--min-patient-coverage", "0.20",
    ]
    print("[run ] " + " ".join(cmd))
    rc = subprocess.call(cmd)
    if rc != 0:
        raise RuntimeError(f"build_prediction_inputs failed for {run['label']} with rc={rc}")


def cohort_diagnostics(run):
    print(f"\n========== cohort diagnostics: {run['title']} ==========")
    for lm in LANDMARKS:
        agg_path = run["inputs_dir"] / f"aggregated_landmark{lm}.csv"
        if not agg_path.exists():
            print(f"  landmark +{lm}d: aggregated CSV not found, skipping")
            continue
        agg = pd.read_csv(agg_path)

        def find_col(substr, stat):
            return next(
                (c for c in agg.columns if substr.lower() in c.lower() and c.endswith(f"__{stat}")),
                None,
            )

        n_plat = int(agg["PLATINUM"].sum())
        print(f"=== landmark +{lm}d | n_total={len(agg):,} n_PLATINUM={n_plat} ===")
        for lab_substr in ("Testosterone", "PSA", "Prostate specific Ag"):
            for stat in ("mean", "last", "max", "min"):
                col = find_col(lab_substr, stat)
                if col is None:
                    continue
                for ev in (0, 1):
                    sub = agg.loc[agg["PLATINUM"] == ev, col].dropna()
                    if sub.empty:
                        continue
                    print(
                        f"  {lab_substr:>22s} {stat:5s} PLAT={ev}: median={sub.median():>10.2f} "
                        f"max={sub.max():>12.2f} n={len(sub):>5}"
                    )
                break
        print()


def build_model_command(model, landmark, config_dir, row_output_dir, run):
    if model == "univariate":
        return [
            PYTHON, str(SURVIVAL_DIR / "univariate_analysis.py"),
            "--inputs-dir", str(run["inputs_dir"]),
            "--output-dir", str(row_output_dir),
            "--landmark-days", str(landmark),
            "--endpoints", "platinum",
        ]
    if model == "elastic-net":
        cmd = [
            PYTHON, str(SURVIVAL_DIR / "multivariate_analysis.py"),
            "--model", "elastic-net",
            "--inputs-dir", str(run["inputs_dir"]),
            "--output-dir", str(row_output_dir),
            "--landmark-days", str(landmark),
            "--endpoints", "platinum",
            "--n-folds", str(N_FOLDS),
        ]
        if config_dir == "baseline":
            cmd.append("--baseline")
        return cmd
    if model == "xgboost":
        cmd = [
            PYTHON, str(SURVIVAL_DIR / "multivariate_analysis.py"),
            "--model", "xgboost",
            "--inputs-dir", str(run["inputs_dir"]),
            "--output-dir", str(row_output_dir),
            "--landmark-days", str(landmark),
            "--endpoints", "platinum",
            "--n-folds", str(N_FOLDS),
        ]
        if config_dir == "baseline":
            cmd.append("--baseline")
        return cmd
    raise ValueError(f"Unknown model: {model}")


def run_models(run):
    print(f"\n========== run models: {run['title']} ==========")
    summary = []
    for model, landmark, config_dir, metrics_filename in TASKS:
        row_output_dir = run["output_dir"] / model_output_dir(model) / f"landmark_{landmark}" / config_dir
        metrics_path = row_output_dir / metrics_filename
        tag = f"{run['label']:16s} {model:11s} landmark_{landmark:<2} {config_dir}"
        if metrics_path.exists() and not FORCE_RERUN:
            print(f"[skip] {tag} -> {metrics_path.relative_to(run['output_dir'])} exists")
            summary.append((tag, "skipped", 0.0))
            continue
        row_output_dir.mkdir(parents=True, exist_ok=True)
        cmd = build_model_command(model, landmark, config_dir, row_output_dir, run)
        print(f"[run ] {tag}")
        print("       " + " ".join(cmd))
        t0 = time.time()
        rc = subprocess.call(cmd)
        elapsed = time.time() - t0
        status = "ok" if rc == 0 else f"FAILED (rc={rc})"
        print(f"[done] {tag} -> {status} ({elapsed/60:.1f} min)\n")
        summary.append((tag, status, elapsed))
    print("\n=== run summary ===")
    for tag, status, elapsed in summary:
        print(f"  {tag} {status:>20s} {elapsed/60:6.1f} min")
    return summary


def summarize_outputs(run):
    rows = []
    for model, landmark, config_dir, metrics_filename in TASKS:
        if model == "univariate":
            continue
        metrics_path = run["output_dir"] / model_output_dir(model) / f"landmark_{landmark}" / config_dir / metrics_filename
        base = {"run": run["label"], "model": model, "landmark": landmark, "config": config_dir, "endpoint": "platinum"}
        if not metrics_path.exists():
            rows.append({**base, "n_test": None, "n_test_events": None, "c_index": None,
                         "mean_auc_t": None, "integrated_brier": None, "status": "missing"})
            continue
        df = pd.read_csv(metrics_path)
        platinum = df.loc[df["endpoint"] == "platinum"]
        if platinum.empty:
            rows.append({**base, "n_test": None, "n_test_events": None, "c_index": None,
                         "mean_auc_t": None, "integrated_brier": None, "status": "no platinum row"})
            continue
        platinum = platinum.iloc[0]
        if model == "elastic-net":
            rows.append({
                **base,
                "n_test": int(platinum["n_test"]),
                "n_test_events": int(platinum["n_events_test"]),
                "c_index": float(platinum["test_c_index"]),
                "mean_auc_t": float(platinum["test_mean_auc_t"]),
                "integrated_brier": float(platinum["test_integrated_brier"]),
                "status": "ok",
            })
        elif model == "xgboost":
            rows.append({
                **base,
                "n_test": int(platinum["n_test"]),
                "n_test_events": int(platinum["n_test_events"]),
                "c_index": float(platinum["c_index"]),
                "mean_auc_t": float(platinum["mean_auc_t"]),
                "integrated_brier": float(platinum["integrated_brier"]),
                "status": "ok",
            })
    return pd.DataFrame(rows).sort_values(["run", "landmark", "model", "config"]).reset_index(drop=True)


## 4. Treatment anchor

In [ ]:
tx_anchor = RUNS[0]
build_prediction_inputs(tx_anchor)
cohort_diagnostics(tx_anchor)
tx_anchor_run_summary = run_models(tx_anchor)
tx_anchor_summary_df = summarize_outputs(tx_anchor)
tx_anchor_summary_df


## 5. Summary

In [ ]:
tx_anchor_summary_df